In [2]:
import os
import cv2
import albumentations as A
from albumentations import HorizontalFlip, RandomBrightnessContrast

# Define paths
input_folder = r"E:\Projects\Strabismus\Split_Data\train"  # Change this to your local dataset path
output_folder = "E:\Projects\Strabismus\Final_Train_data"  # Change this to where you want augmented images

# Ensure the output folder exists
os.makedirs(output_folder, exist_ok=True)

# Define augmentations
brightness_augment = RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.0, p=1.0)
contrast_augment = RandomBrightnessContrast(brightness_limit=0.0, contrast_limit=0.2, p=1.0)
horizontal_flip = HorizontalFlip(p=1.0)

# Function to convert to black-and-white
def to_grayscale(image):
    return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Function to apply augmentations
def augment_image(image_path, output_subfolder):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error reading {image_path}. Skipping...")
        return

    base_name = os.path.splitext(os.path.basename(image_path))[0]  # Get filename without extension

    # Save the original image
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_original.jpg"), image)

    # Apply horizontal flip
    flipped = horizontal_flip(image=image)["image"]
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_flipped.jpg"), flipped)

    # Apply brightness adjustments
    for i in range(2):  
        brightness_aug = brightness_augment(image=image)["image"]
        cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_brightness_{i + 1}.jpg"), brightness_aug)

    flipped_brightness = brightness_augment(image=flipped)["image"]
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_flipped_brightness.jpg"), flipped_brightness)

    # Apply contrast adjustments
    for i in range(2):  
        contrast_aug = contrast_augment(image=image)["image"]
        cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_contrast_{i + 1}.jpg"), contrast_aug)

    flipped_contrast = contrast_augment(image=flipped)["image"]
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_flipped_contrast.jpg"), flipped_contrast)

    # Convert to grayscale
    grayscale_original = to_grayscale(image)
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_grayscale.jpg"), grayscale_original)

    grayscale_flipped = to_grayscale(flipped)
    cv2.imwrite(os.path.join(output_subfolder, f"{base_name}_flipped_grayscale.jpg"), grayscale_flipped)

# Process each class folder
for subfolder_name in os.listdir(input_folder):
    subfolder_path = os.path.join(input_folder, subfolder_name)
    if os.path.isdir(subfolder_path):  
        output_subfolder = os.path.join(output_folder, subfolder_name)
        os.makedirs(output_subfolder, exist_ok=True)

        # Process each image
        for image_name in os.listdir(subfolder_path):
            image_path = os.path.join(subfolder_path, image_name)
            if os.path.isfile(image_path) and image_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                augment_image(image_path, output_subfolder)

print("Image augmentation complete. Check the output folder.")


Image augmentation complete. Check the output folder.
